# Begin

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import itertools
import json
import datetime
import pprint
from functools import cache
import re
import pickle
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

from tqdm.notebook import tqdm

import lark # @launchit.collect

import numpy as np
import einops
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import DataLoader, StackDataset
from torchvision import datasets

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from utils import * # @launchit.collect
from logging_utils import *
from model_registry import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect_2
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement
from torch_helpers import *

# Setup

In [3]:
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    private_data_path=None,
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'subproject_path': '/home/misha/dev/mine/neurovision/12_rnn',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'private_data_path': '/home/misha/dev/mine/neurovision/data/12_rnn',
 'run_path': '/home/misha/dev/mine/neurovision/run/12_rnn',
 'self_fname': '/home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01.ipynb',
 'self_name': 'gen_text_rnn_01',
 'subproject_name': '12_rnn',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



In [4]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()

class LaunchGoalObj(namedtuple('LaunchGoalObj', 'goal, model_group_uri, model_name, model_version, model_main_asset_fname')):
    def matches(self, *g):
        if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
            return True
            
        if g and isinstance(g[0], list):
            assert len(g) == 1
            return self.goal in g[0]

        return self.goal in g

LAUNCH_GOAL = LaunchGoalObj(
    goal=LangUtils.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED),
    model_group_uri='${MODEL_GROUP_URI}',
    model_name='${MODEL_NAME}',
    model_version=LangUtils.from_str(int, '${MODEL_VERSION}', 0),
    model_main_asset_fname='${LAUNCHIT_FNAME}',
)
# @launchit.stop

LAUNCH_GOAL = LAUNCH_GOAL._replace(goal=LaunchGoal.UNSPECIFIED)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_name=CONFIG.self_name)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_version=0)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_main_asset_fname=CONFIG.self_fname)
# @launchit.stop

LOG(f'LAUNCH_GOAL=\n{pprint.pformat(LAUNCH_GOAL._asdict(), sort_dicts=False)}', when=CONFIG.is_interactive)
LOG(f'LAUNCH_GOAL={LAUNCH_GOAL._asdict()}', when=not CONFIG.is_interactive)

LAUNCH_GOAL=
{'goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'model_group_uri': 'com.develorium.neurovision.12_rnn',
 'model_name': 'gen_text_rnn_01',
 'model_version': 0,
 'model_main_asset_fname': '/home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01.ipynb'}


# Hyperparameters

In [123]:
# @launchit.disable
# @launchit.collect
@dataclass
class Hyperparameters:
    random_seed: int = None
    # input data params
    chunk_size: int = None
    # model structure params
    embedding_size: int = None
    state_model_units: list[str] = None 
    lr_model_units: list[str] = None
    # training params
    batch_size: int = None
    epochs_count: int = None
    optimizer: str = None
    learn_rate: float | str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

HP = Hyperparameters()
HP.random_seed = 42


In [124]:
# grammar = '''
#     spec: lstm_spec # | rnn_spec | gru_spec

#     lstm_spec: "LSTM" "(" LSTM_HIDDEN_SIZE ")" ("x" LSTM_NUM_LAYERS)?
#     LSTM_HIDDEN_SIZE: NUMBER
#     LSTM_NUM_LAYERS: NUMBER
    
#     %import common.WORD
#     %import common.NUMBER
#     %import common.WS
#     %ignore WS
# '''
# parser = lark.Lark(grammar, start='spec')
# print(parser.parse('LSTM(64)x2').pretty())

In [129]:
@dataclass
class StateModelUnitParams: 
    module: str = None
    hidden_size: int = None
    num_layers: int = None

@dataclass
class LogRegModelUnitParams: 
    items_count: int = None
    nonlinearity: str = None # any of: None, sigmoid, tanh, ...
    dropout: float = None

@dataclass
class LearnRateParams: 
    Plateau = namedtuple('Plateau', 'factor patience')
    
    learn_rate: float = None
    plateau: object = None

def get_lark_tree_value(tree, var_name, default_value=None):
    try:
        return next(tree.scan_values(lambda i: i.type == var_name)).value
    except StopIteration:
        return default_value

def hp_parse_state_model_units(self):
    grammar = '''
        spec: lstm_spec # | rnn_spec | gru_spec
    
        lstm_spec: "LSTM" "(" LSTM_HIDDEN_SIZE ")" ("x" LSTM_NUM_LAYERS)?
        LSTM_HIDDEN_SIZE: NUMBER
        LSTM_NUM_LAYERS: NUMBER
        
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.state_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = StateModelUnitParams()

        if list(tree.find_data('lstm_spec')):
            params.module = 'LSTM'
            params.hidden_size = int(gtv('LSTM_HIDDEN_SIZE'))
            params.num_layers = int(gtv('LSTM_NUM_LAYERS', 1))
            
        params_list.append(params)

    return params_list

def hp_parse_lr_model_units(self):
    grammar = '''
        spec: (dropout_spec "->")? items_count_spec ("->" nonlinearity_spec)?
    
        dropout_spec: "dropout" "(" DROPOUT_P ")"
        DROPOUT_P: NUMBER
    
        items_count_spec: ITEMS_COUNT
        ITEMS_COUNT: NUMBER
    
        nonlinearity_spec: NONLINEARITY_WORD
        NONLINEARITY_WORD: WORD
    
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.lr_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = LogRegModelUnitParams()
        params.items_count = int(gtv('ITEMS_COUNT'))
        params.nonlinearity = gtv('NONLINEARITY_WORD')
        params.dropout = float(gtv('DROPOUT_P')) if gtv('DROPOUT_P', False) else None
        params_list.append(params)

    return params_list

def hp_parse_learn_rate(self):
    params = LearnRateParams()

    if isinstance(self.learn_rate, float):
        params.learn_rate = self.learn_rate
        return params
        
    grammar = '''
        spec: initial_lr_spec(","plateau_spec)?
    
        initial_lr_spec: INITIAL_LR
        INITIAL_LR: NUMBER
    
        plateau_spec: "plateau" "(" (|plateau_params_spec ("," plateau_params_spec)*) ")"
        plateau_params_spec: plateau_factor_spec | plateau_patience_spec
        plateau_factor_spec: "factor" "=" plateau_factor_value_spec
        plateau_factor_value_spec: PLATEAU_FACTOR_VALUE
        plateau_patience_spec: "patience" "=" plateau_patience_value_spec
        plateau_patience_value_spec: PLATEAU_PATIENCE_VALUE
        PLATEAU_FACTOR_VALUE: NUMBER
        PLATEAU_PATIENCE_VALUE: NUMBER
        
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    tree = parser.parse(self.learn_rate)
    gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
    params.learn_rate = float(gtv('INITIAL_LR'))
    
    if list(tree.find_data('plateau_spec')):
        params.plateau = LearnRateParams.Plateau(
            factor=float(gtv('PLATEAU_FACTOR_VALUE', 0.1)),
            patience=int(gtv('PLATEAU_PATIENCE_VALUE', 10)),
        )

    return params

Hyperparameters.parse_state_model_units = hp_parse_state_model_units
Hyperparameters.parse_lr_model_units = hp_parse_lr_model_units
Hyperparameters.parse_learn_rate = hp_parse_learn_rate

In [133]:
# @launchit.disable
x = Hyperparameters(state_model_units=['LSTM(64)x10', 'LSTM(32)'])
x.parse_state_model_units()

[StateModelUnitParams(module='LSTM', hidden_size=64, num_layers=10),
 StateModelUnitParams(module='LSTM', hidden_size=32, num_layers=1)]

In [132]:
# @launchit.disable
x = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    '200->sigmoid',
])
x.parse_lr_model_units()

[LogRegModelUnitParams(items_count=200, nonlinearity='tanh', dropout=0.5),
 LogRegModelUnitParams(items_count=200, nonlinearity='sigmoid', dropout=None)]

In [131]:
# @launchit.disable
x = Hyperparameters(learn_rate='0.005,plateau(factor=0.115, patience=15)')
lr_params = x.parse_learn_rate()
o = torch.optim.Adam(params=[nn.Parameter(torch.tensor(1.))])
p = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=o, **lr_params.plateau._asdict())
lr_params, p.factor, p.patience

(LearnRateParams(learn_rate=0.005, plateau=Plateau(factor=0.115, patience=15)),
 0.115,
 15)

# Launch

## new_model_registry

In [8]:
def new_model_registry(is_real=None, group_uri=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ModelRegistry(LAUNCH_GOAL.model_group_uri if group_uri is None else group_uri)

## new_summary_writer

In [9]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Bootstrap

In [10]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', LAUNCH_GOAL.model_version)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if True or LAUNCH_GOAL.goal != LaunchGoal.TRAIN_MODEL_AFTER_CV:
    model_registry = new_model_registry()
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, CONFIG.self_fname, replace=True)
        
    meta = dict(
        optuna_trial_number=getattr(optuna_trial, 'number', None),
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = LAUNCH_GOAL.model_name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(LAUNCH_GOAL.model_version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=42
Tensorboard run=gen_text_rnn_01/0


<Mock name='mock.add_text()' id='124489900580624'>

In [11]:
mr = new_model_registry(is_real=True, group_uri='org.gutenberg')
# mr.get_asset_content('ulysses', 1, 'txt')
# mr.get_assets('The*Great*Gatsby', 1)

# Dataset

In [12]:
# @launchit.disable
# @launchit.collect
HP.chunk_size = 20

## TextPreprocessor

In [13]:
class TextPreprocessor:
    @staticmethod
    def remove_pg_envelope(text):
        start_indx = text.find('START OF THIS PROJECT')
        start_indx = text.find('START OF THE PROJECT') if start_indx < 0 else start_indx
        end_indx = text.find('End of the Project Gutenberg')
        end_indx = text.find('END OF THE PROJECT GUTENBERG') if end_indx < 0 else end_indx
        assert start_indx >= 0
        assert end_indx >= 0
        
        while text[start_indx] != '\n': 
            start_indx += 1
        
        while text[start_indx] == '\n':
            start_indx += 1
        
        while text[end_indx] != '\n':
            end_indx -= 1

        return text[start_indx:end_indx]

    @staticmethod
    def preprocess(text):
        text = re.sub(r'[“”]', '"', text)
        text = re.sub(r'[‘’]', "'", text)
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'([^\n])\n([^\n])', r'\1 \2', text)
        text = re.sub(r'\n(\n)+', '\n', text)
        text = re.sub(r'-(-)+', '. ', text)
        # text = re.sub(r'[^—]—[^—]', ' — ', text)
        text = re.sub(r'\.\.\.', '…', text)
        text = text.lower()
        return text

    SPLIT_CHARS = r'"\',.:;?!()\[\]=\+\*/—…\-\$_'
    
    @staticmethod
    def split_line(line):
        r = []

        # Combination of line.split() and re.split() is used to retain separators as individual tokens
        # Note: re.split with grouping is a must - in opposite case separators will get lost
        for x in line.split():
            r.extend(filter(None, re.split(r'([' + TextPreprocessor.SPLIT_CHARS + r'])', x)))

        return r

    @staticmethod
    def check_token(token):
        return re.match(r'[\w' + TextPreprocessor.SPLIT_CHARS +  r']+', token)

## Bundle

In [14]:
@dataclass
class Bundle:
    df_sources: object = None
    df_vocab: object = None
    df_chunks: object = None

    @staticmethod
    def get_fname():
        fname = f'bundle_{HP.chunk_size}.pkl' 
        return os.path.join(CONFIG.private_data_path, fname)

## Generate

In [98]:
# @launchit.disable
assert CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK
fname = Bundle.get_fname()

if not os.path.exists(fname):
    LOG(f'"{fname}" not found, generating new one')
    preprocessed_texts_dir_name = os.path.join(CONFIG.private_data_path, 'preprocessed')
    os.makedirs(preprocessed_texts_dir_name, exist_ok=True)
    
    sources = [
        'crime_and_punishment',
        'the_great_gatsby',
        'the_mysterious_island',
        'the_picture_of_dorian_gray',
        'ulysses',
    ]
    source_to_text = {}
    source_to_tokens = {}
    df_sources = pd.DataFrame(dict(
        name=sources,
        fname=list(map(lambda s: os.path.join(preprocessed_texts_dir_name, s + '.txt'), sources)),
    ))

    for row in df_sources.itertuples():
        mr = new_model_registry(is_real=True, group_uri='org.gutenberg')
        text = mr.get_asset_content(row.name, 1, 'txt').decode('utf-8')
        text = TextPreprocessor.remove_pg_envelope(text)
        text = TextPreprocessor.preprocess(text)

        with open(row.fname, 'w', encoding="utf-8") as f:
            f.write(text)

        source_to_text[row.name] = text

    vocab = Counter()
    chunks = []

    for iter_no, iter_name in enumerate(('Vocab', 'Tokens', 'Chunks')):
        if iter_no in [0, 1]:
            for source_name, text in source_to_text.items():
                lines = text.splitlines()
                tokens = [] # for each line will contain list of tokens, i.e. list of lists
                
                for line_ind, line in tqdm(enumerate(lines), total=len(lines), desc=f'{iter_name}, {source_name}'):
                    line_tokens = TextPreprocessor.split_line(line)
        
                    # if not line_tokens:
                    #     continue
    
                    if iter_no == 0:
                        vocab.update(line_tokens)
                    else:
                        tokens.append(list(map(lambda t: vocab[t][0], line_tokens)))
                        
                source_to_tokens[source_name] = tokens
    
            if iter_no == 0:
                text_list, freq_list = zip(*vocab.most_common())
                ind_list = list(range(len(text_list)))
                vocab = dict(zip(text_list, zip(ind_list, freq_list))) # make dict: text -> (ind, freq)
        
        elif iter_no == 2:
            for source_name, tokens in source_to_tokens.items():
                source_ind = df_sources[df_sources.name == source_name].index.item()
                line_inds = [] # aligned with tokens (1-to-1 mapping), stamps every token with a line index

                for line_ind, line_tokens in enumerate(tokens):
                    line_inds.extend([line_ind] * len(line_tokens))

                tokens = list(itertools.chain.from_iterable(tokens)) # flatten tokens list
                assert len(line_inds) == len(tokens)
                
                for start_ind in tqdm(range(len(tokens) - HP.chunk_size), desc=f'{iter_name}, {source_name}'):
                    chunk = tokens[start_ind:start_ind+HP.chunk_size]
                    assert len(chunk) == HP.chunk_size
                    chunks.append((np.array(chunk), source_ind, line_inds[start_ind], line_inds[start_ind+HP.chunk_size-1]))
        
        else:
            assert False, (iter_no, iter_name)
    
    text_list, ind_freq_list = zip(*vocab.items())
    ind_list, freq_list = zip(*ind_freq_list)
    df_vocab = pd.DataFrame(dict(
        text=text_list,
        freq=freq_list,
    ), index=ind_list)
    chunk_list, source_id_list, start_line_ind_list, end_line_ind_list = zip(*chunks)
    df_chunks = pd.DataFrame(dict(
        chunk=chunk_list,
        source_ind=source_id_list,
        start_line_ind=start_line_ind_list,
        end_line_ind=end_line_ind_list,
    ))
    bundle = Bundle(df_sources=df_sources, df_vocab=df_vocab, df_chunks=df_chunks)

    with open(fname, 'wb') as f:
        pickle.dump(bundle, f)
        
    LOG(f'Bundle generated and saved to "{fname}"')

"/home/misha/dev/mine/neurovision/data/12_rnn/bundle_20.pkl" not found, generating new one


Vocab, crime_and_punishment:   0%|          | 0/3970 [00:00<?, ?it/s]

Vocab, the_great_gatsby:   0%|          | 0/1653 [00:00<?, ?it/s]

Vocab, the_mysterious_island:   0%|          | 0/4905 [00:00<?, ?it/s]

Vocab, the_picture_of_dorian_gray:   0%|          | 0/1527 [00:00<?, ?it/s]

...

Chunks, the_great_gatsby:   0%|          | 0/61718 [00:00<?, ?it/s]

Chunks, the_mysterious_island:   0%|          | 0/233950 [00:00<?, ?it/s]

Chunks, the_picture_of_dorian_gray:   0%|          | 0/97500 [00:00<?, ?it/s]

Chunks, ulysses:   0%|          | 0/329240 [00:00<?, ?it/s]

Bundle generated and saved to "/home/misha/dev/mine/neurovision/data/12_rnn/bundle_20.pkl"


## Load

In [99]:
bundle_fname = Bundle.get_fname()

with open(bundle_fname, 'rb') as b:
    bundle = pickle.load(b)

chunks = np.array(bundle.df_chunks['chunk'].values.tolist())
assert chunks.shape[1] == HP.chunk_size
LOG(f'Bundle with chunks loaded from "{bundle_fname}"')

Bundle with chunks loaded from "/home/misha/dev/mine/neurovision/data/12_rnn/bundle_20.pkl"


## Verify

In [107]:
# @launchit.disable
columns = defaultdict(list)
chunks_to_show = 10
chunk_inds = RNG.choice(len(bundle.df_chunks), 10, replace=False)
df_inspect = bundle.df_chunks.loc[chunk_inds]
df_inspect = pd.merge(df_inspect, bundle.df_sources, left_on='source_ind', right_index=True, how='left')
df_inspect.rename(columns={'name': 'source_name', 'fname': 'source_fname'}, inplace=True)
df_inspect['chunk1'] = df_inspect['chunk'].map(lambda c: list(map(lambda t: bundle.df_vocab.iloc[t].text, c)))

sources = {}

for row in df_inspect.itertuples():
    print(f'Chunk #{row.Index}, source={row.source_name} (#{row.source_ind}), lines={row.start_line_ind}:{row.end_line_ind}')
    print('- as int list: ' + ' '.join(map(str, row.chunk)))
    print('- as str list: ' + ' '.join(map(str, row.chunk1)))
    
    if not row.source_name in sources:
        with open(row.source_fname, 'r', encoding='utf-8') as f:
            sources[row.source_name] = f.readlines()
        
    print('- origin:      ' + ''.join(sources[row.source_name][row.start_line_ind:row.end_line_ind+1]))
    print()

Chunk #900176, source=ulysses (#4), lines=6167:6169
- as int list: 63 15 222 43 7 146 7 384 0 11031 54 111 480 42 229 3 1348 1 25 67
- as str list: if you give me a hand a second , sergeant … first watch : name and address . _ (
- origin:      bloom: _(peering over the crowd.)_ i just see a car there. if you give me a hand a second, sergeant…
first watch: name and address.
_(corny kelleher, weepers round his hat, a death wreath in his hand, appears among the bystanders.)_


Chunk #765758, source=ulysses (#4), lines=3580:3580
- as int list: 91 2616 2292 10819 535 50 3511 91 6186 70 11 14 9 2680 5019 10 20 2293 104 9
- as str list: two sheets cream vellum paper one reserve two envelopes when i was in wisdom hely ' s wise bloom in
- origin:      two sheets cream vellum paper one reserve two envelopes when i was in wisdom hely's wise bloom in daly's henry flower bought. are you not happy in your home? flower to console me and a pin cuts lo. means something, language of flow. was it a daisy

# Model

## StateModelUnit

In [182]:
class StateModelUnit(nn.Module):
    def __init__(self, input_size, unit_hp):
        super().__init__()

        match unit_hp.module:
            case 'LSTM':
                self.module = nn.LSTM(input_size, unit_hp.hidden_size, num_layers=unit_hp.num_layers, batch_first=True)
            case _:
                assert False, f'Unsupported {unit_hp.module=}'

        self.output_size = self.module.hidden_size

    def forward(self, inp): 
        # inp.shape: batch, chunk, input (of embedding_size or of ouput_size of preceeding unit)
        res, (hidden, cell) = self.module(inp)
        return res

## StateModel

In [183]:
class StateModel(nn.Module):
    def __init__(self, input_size, model_hp):
        super().__init__()
        self.units = nn.Sequential()

        for unit_hp in model_hp.parse_state_model_units():
            unit = StateModelUnit(input_size, unit_hp)
            self.units.append(unit)
            input_size = unit.output_size

        self.output_size = self.units[-1].output_size

    def forward(self, inp):
        if self.training:
            return self.units(inp)
        else:
            with torch.no_grad():
                return self.units(inp)

In [185]:
# @launchit.disable
embedding_size = 128
model_hp = Hyperparameters(state_model_units=[
    'LSTM(64)x5',
    'LSTM(32)x3'
])
model = StateModel(embedding_size, model_hp)
probe_tensor = torch.ones(1, 10, embedding_size)
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

StateModel(
  (units): Sequential(
    (0): StateModelUnit(
      (module): LSTM(128, 64, num_layers=5, batch_first=True)
    )
    (1): StateModelUnit(
      (module): LSTM(64, 32, num_layers=3, batch_first=True)
    )
  )
)
Parameters count=212224
Output shape=torch.Size([1, 10, 32])


## LogRegModelUnit

In [186]:
class LogRegModelUnit(nn.Module):
    def __init__(self, input_dims_count, unit_hp):
        super().__init__()
        self.linear = nn.Linear(
            in_features=input_dims_count, 
            out_features=unit_hp.items_count, 
            bias=True
        )

        if unit_hp.nonlinearity is None:
            self.nonlinearity = lambda i: i
        else:
            self.nonlinearity = getattr(F, unit_hp.nonlinearity)

        if unit_hp.dropout is None:
            self.dropout = lambda i: i
        else:
            assert unit_hp.dropout >= 0
            self.dropout = nn.Dropout(unit_hp.dropout)

        self.output_size = self.linear.out_features

    def forward(self, inp): 
        res = self.dropout(inp)
        res = self.linear(res)
        res = self.nonlinearity(res)
        return res

## LogRegModel

In [187]:
class LogRegModel(nn.Module):
    def __init__(self, input_size, model_hp):
        super().__init__()
        self.units = nn.Sequential()

        for unit_hp in model_hp.parse_lr_model_units():
            unit = LogRegModelUnit(input_size, unit_hp)
            self.units.append(unit)
            input_size = unit.output_size

        self.output_size = self.units[-1].output_size

    def forward(self, inp):
        return self.units(inp)

In [188]:
# @launchit.disable
input_size = 128
model_hp = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    '80->sigmoid',
])
model = LogRegModel(input_size, model_hp)
probe_tensor = torch.ones(1, 10, input_size)
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

LogRegModel(
  (units): Sequential(
    (0): LogRegModelUnit(
      (linear): Linear(in_features=128, out_features=200, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
    )
    (1): LogRegModelUnit(
      (linear): Linear(in_features=200, out_features=80, bias=True)
    )
  )
)
Parameters count=41880
Output shape=torch.Size([1, 10, 80])


## MainModel

In [196]:
class MainModel(nn.Module):
    def __init__(self, vocab_size, model_hp):
        super().__init__()
        self.pipeline = nn.Sequential()
        self.pipeline.append(nn.Embedding(vocab_size, model_hp.embedding_size))
        state_model = StateModel(model_hp.embedding_size, model_hp)
        self.pipeline.append(state_model)
        lr_model = LogRegModel(state_model.output_size, model_hp)
        self.pipeline.append(lr_model)
        self.pipeline.append(nn.Linear(lr_model.output_size, vocab_size))
        self.output_size = vocab_size

    def forward(self, inp):
        return self.pipeline(inp)

In [199]:
# @launchit.disable
vocab_size = 10_000
model_hp = Hyperparameters(
    embedding_size=256,
    state_model_units=[
        'LSTM(64)x5',
        'LSTM(32)x3',
        ],
    lr_model_units=[
        'dropout(0.5)->200->tanh',
        '80->sigmoid',
    ],
)
model = MainModel(vocab_size, model_hp)
probe_tensor = torch.ones(2, 10, dtype=int) # batch of 2 sample each containing 10 sequences (chunks, eachk elem of chunk is a token)
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
sum([p.numel() for p in model.parameters()])
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

MainModel(
  (pipeline): Sequential(
    (0): Embedding(10000, 256)
    (1): StateModel(
      (units): Sequential(
        (0): StateModelUnit(
          (module): LSTM(256, 64, num_layers=5, batch_first=True)
        )
        (1): StateModelUnit(
          (module): LSTM(64, 32, num_layers=3, batch_first=True)
        )
      )
    )
    (2): LogRegModel(
      (units): Sequential(
        (0): LogRegModelUnit(
          (linear): Linear(in_features=32, out_features=200, bias=True)
          (dropout): Dropout(p=0.5, inplace=False)
        )
        (1): LogRegModelUnit(
          (linear): Linear(in_features=200, out_features=80, bias=True)
        )
      )
    )
    (3): Linear(in_features=80, out_features=10000, bias=True)
  )
)
Parameters count=3637672
Output shape=torch.Size([2, 10, 10000])


In [198]:
# class StateModel(nn.Module):
#     def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
#         super().__init__()
#         HP param for embedding
#         self.embedding = nn.Embedding(vocab_size, embed_dim) 
        
#         self.rnn_hidden_size = rnn_hidden_size

#         HP params / DSL for rnn layers (LSTM, GRU)
#         num_layers
        
#         self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, 
#                            batch_first=True)

#         self.fc = nn.Linear(rnn_hidden_size, vocab_size)

#     def forward(self, x, hidden, cell):
#         out = self.embedding(x).unsqueeze(1)
#         out, (hidden, cell) = self.rnn(out, (hidden, cell))
#         out = self.fc(out).reshape(out.size(0), -1)
#         return out, hidden, cell

#     def init_hidden(self, batch_size):
#         hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
#         cell = torch.zeros(1, batch_size, self.rnn_hidden_size)
#         return hidden.to(device), cell.to(device)

In [111]:
assert False

AssertionError: 

# Model training via CV

Build number of model instances, train each on separate cross-validation fold from training dataset => train and validation metrics

## Configure

In [27]:
# @launchit.disable
# @launchit.collect_1
HP.fe_model_units = [
    'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); from:s4_psd_01,450',
    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
]
HP.lr_model_units = [
    '200->tanh',
    '10->tanh',
]
HP.refine_fe_model = True
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.learn_rate = 0.001
HP.cv_folds_count = 5
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 42,
 'images_preprocessing': 'SAMPLE_WISE',
 'fe_model_units': ['conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); '
                    'from:04_psd_01,396',
                    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'],
 'lr_model_units': ['200->tanh', '10->tanh'],
 'refine_fe_model': True,
 'batch_size': 100,
 'epochs_count': 1,
 'optimizer': 'Adam',
 'learn_rate': 0.001,
 'cv_folds_count': 5}


## Train

In [126]:
# @launchit.disable_1
if LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL_CV):
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
    k_fold = KFold(HP.cv_folds_count, shuffle=True, random_state=HP.random_seed)
    metrics_suite = defaultdict(list)
    optuna_trial = optuna_multiprocessing.get_trial()
    dataset, _ = create_dataset_for_supervised_training('TRAIN')
    
    for fold_ind, (train_inds, val_inds) in tqdm(enumerate(k_fold.split(dataset)), desc='Fold', total=HP.cv_folds_count, disable=not CONFIG.is_interactive):
        with LOG.auto_prefix('FOLD', fold_ind):
            data_loader = DataLoader(StackDataset(*dataset[train_inds]), batch_size=HP.batch_size, shuffle=True)
            val_images = dataset[val_inds][0].to(device=CONFIG.cuda_device)
            val_images = einops.rearrange(val_images, 'b h w -> b 1 h w')
            val_labels = dataset[val_inds][1].to(device=CONFIG.cuda_device)
    
            if HP.refine_fe_model:
                this_fe_model = copy.deepcopy(fe_model)
                this_fe_model.train()
            else:
                this_fe_model = fe_model
                this_fe_model.eval()
                
            model = MainModel(dataset[0][0].shape, this_fe_model.to(device='cpu'), HP)
            model = model.to(device=CONFIG.cuda_device)
            loss_fn = nn.CrossEntropyLoss()
            lr_params = HP.parse_learn_rate()
            optimizer = getattr(torch.optim, HP.optimizer)(model.parameters(), lr=lr_params.learn_rate)
            lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)
            fold_metrics_suite = defaultdict(list)
        
            for epoch_ind in tqdm(range(HP.epochs_count), 'Epoch', disable=not CONFIG.is_interactive):
                epoch = epoch_ind + 1
                
                with LOG.auto_prefix('EPOCH', epoch):
                    train_loss_sum, train_loss_denom = 0, 0
                    train_accuracy_sum, train_accuracy_denom = 0, 0
                    
                    for batch in data_loader:
                        optimizer.zero_grad()
                        
                        images = batch[0].to(device=CONFIG.cuda_device)
                        images = einops.rearrange(images, 'b h w -> b 1 h w')
                        labels = batch[1].to(device=CONFIG.cuda_device)
                        logits = model(images)
                        loss = loss_fn(logits, labels)
                        loss.backward()
                        
                        optimizer.step()
                        
                        train_loss_sum += loss.item() * len(batch)
                        train_loss_denom += len(batch)
                        train_accuracy_sum += (logits.argmax(axis=1) == labels).sum().item() # how many correct predictions
                        train_accuracy_denom += len(labels)
                
                    assert train_loss_denom > 0
                    assert train_accuracy_denom > 0
                    train_loss = train_loss_sum / train_loss_denom
                    train_accuracy = train_accuracy_sum / train_accuracy_denom
                    fold_metrics_suite['train_loss'].append(train_loss)
                    fold_metrics_suite['train_accuracy'].append(train_accuracy)
                    
                    lr_scheduler.step(train_loss)
                
                    with torch.no_grad():
                        val_logits = model(val_images)
                        val_loss = loss_fn(val_logits, val_labels).item()
                        predicted_val_labels = val_logits.argmax(axis=1)
                        val_accuracy = (predicted_val_labels == val_labels).sum().item() / len(val_labels)
                        fold_metrics_suite['val_loss'].append(val_loss)
                        fold_metrics_suite['val_accuracy'].append(val_accuracy)
    
                    LOG(f'{train_loss=:.2f}, {train_accuracy=:.2f}, {val_loss=:.2f}, {val_accuracy=:.2f}', when=not CONFIG.is_interactive)
                    
                    if optuna_trial is not None and fold_ind == 0:
                        # https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html#optuna.trial.Trial.report:
                        # > If this method is called multiple times at the same step in a trial, the reported value only the first time is stored 
                        # > and the reported values from the second time are ignored.
                        # In other words calling report for fold other than the first one does nothing except producing tons of warnings in console. 
                        # As such only the first fold is indicative
                        optuna_trial.report(val_accuracy, epoch_ind) # "Note that pruners assume that step starts at zero" -> as such use epoch_ind instead of epoch
                
                        if optuna_trial.should_prune():
                            # Despite written in docs OPTUNA_TRIAL.should_prune is not idempotent - consequent calls could lead
                            # to different responses. Perhapse this is due to influence of concurrent trials running which could change
                            # pruner decision. As such cache pruning result so it's immutable
                            optuna_trial.set_user_attr('IS_PRUNED', True)
                            break
        
            for metric_name, metric_values in fold_metrics_suite.items():
                metrics_suite[metric_name].append(metric_values)
        
            if 'IS_PRUNED' in getattr(optuna_trial, 'user_attrs', {}):
                LOG(f'Optuna pruning condition encountered. Stopping training')
                break
        
    for metric_name in list(metrics_suite.keys()):
        metric_values = metrics_suite[metric_name]
        metric_values = np.array(metric_values)
        # According to Optuna docs we could get pruned only on first fold (report() calls is ignored on successive folds).
        # As such we either should have 1) a full-fledged metrcs_suite (for all folds and for each epoch)
        # or 2) only metrics from the very first fold
        if 'IS_PRUNED' in getattr(optuna_trial, 'user_attrs', {}):
            assert len(metric_values) == 1
            assert metric_values.shape[1] <= HP.epochs_count
        else:
            assert len(metric_values) == HP.cv_folds_count
            assert metric_values.shape[1] == HP.epochs_count
            
        mean_values = list(metric_values.mean(axis=0)) # coercing to list since json.dump throws "np.ndarray is not serializable"
        std_values = list(metric_values.std(axis=0)) # coercing to list since json.dump throws "np.ndarray is not serializable"
        metrics_suite['mean_' + metric_name] = mean_values
        metrics_suite['std_' + metric_name] = std_values
        assert len(mean_values) == len(std_values)
        
        for epoch_ind in range(min(HP.epochs_count, len(mean_values))):
            epoch = epoch_ind + 1
            summary_writer.add_scalar('mean_' + metric_name, mean_values[epoch_ind], epoch)
            summary_writer.add_scalar('std_' + metric_name, std_values[epoch_ind], epoch)
            
        summary_writer.flush()   
    
    METRICS_SUITE.update(metrics_suite)

Fold:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

## Save

In [165]:
# @launchit.disable_1
if LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL_CV):
    model_registry = new_model_registry()
    
    with io.BytesIO() as b:
        # Only model from last KFold is saved. A bit senseless...
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'hypers': HP._asdict(),
        }, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='cv', replace=True)
    
    with io.StringIO() as b:
        json.dump(METRICS_SUITE, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

# Model training (test_accuracy)

Build model, train on whole train dataset (without cross validation), validate versus test dataset => test_accuracy.

## Configure

In [104]:
# @launchit.disable
# @launchit.collect_1
HP.fe_model_units = [
    'conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(5,1)->avg(2)',
    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
]
HP.lr_model_units = [
    '200->tanh',
    '10->tanh',
]
HP.refine_fe_model = True
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.learn_rate = 0.001
HP.cv_folds_count = 5
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 42,
 'images_preprocessing': 'SAMPLE_WISE',
 'fe_model_units': ['conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(5,1)->avg(2)',
                    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'],
 'lr_model_units': ['200->tanh', '10->tanh'],
 'refine_fe_model': True,
 'batch_size': 100,
 'epochs_count': 1,
 'optimizer': 'Adam',
 'learn_rate': 0.001,
 'cv_folds_count': 5}


## Load

In [105]:
# @launchit.disable_2
if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
elif LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL):
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
elif LAUNCH_GOAL.matches(TEST_ACCURACY_GOALS):
    model_registry = new_model_registry()
    meta = json.load(io.BytesIO(model_registry.get_asset_content(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, asset_ext='json', asset_classifier='meta')))
    # This launch should EXTEND model storage on Nexus with artifact of trained model on whole dataset. 
    # We lack proper hyperparameters from this launch notebook per se (see corresponding launch creation logic), 
    # so we have to load hyperparameters for model which were used during cross-validation
    HP = Hyperparameters.from_dict(meta['hypers'])
    LOG(f'Hyperparameters loaded from version={LAUNCH_GOAL.model_version}: {HP._asdict()}')
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
    LOG(f'FE model instance loaded from version={LAUNCH_GOAL.model_version}')
    old_metrics_suite = json.load(io.BytesIO(model_registry.get_asset_content(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, asset_ext='json', asset_classifier='metrics')))
    METRICS_SUITE.update(old_metrics_suite)
    LOG(f'Metrics suite loaded from version={LAUNCH_GOAL.model_version}')
else:
    assert False

## Train

In [106]:
# @launchit.disable_2
train_dataset, test_dataset = create_dataset_couple_for_supervised_training()
data_loader = DataLoader(train_dataset, batch_size=HP.batch_size, shuffle=True)

if HP.refine_fe_model:
    this_fe_model = copy.deepcopy(fe_model)
    this_fe_model.train()
else:
    this_fe_model = fe_model
    this_fe_model.eval()

model = MainModel(train_dataset[0][0].shape, this_fe_model.to(device='cpu'), HP)
model = model.to(device=CONFIG.cuda_device)
loss_fn = nn.CrossEntropyLoss()
lr_params = HP.parse_learn_rate()
optimizer = getattr(torch.optim, HP.optimizer)(model.parameters(), lr=lr_params.learn_rate)
lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)

test_images = test_dataset.datasets[0].to(device=CONFIG.cuda_device)
test_images = einops.rearrange(test_images, 'b h w -> b 1 h w')
test_labels = test_dataset.datasets[1].to(device=CONFIG.cuda_device)

for epoch_ind in tqdm(range(HP.epochs_count), 'Epoch', disable=not CONFIG.is_interactive):
    epoch = epoch_ind + 1
    train_loss_sum, train_loss_denom = 0, 0
    train_accuracy_sum, train_accuracy_denom = 0, 0
    
    for batch in data_loader:
        optimizer.zero_grad()
        
        images = batch[0].to(device=CONFIG.cuda_device)
        images = einops.rearrange(images, 'b h w -> b 1 h w')
        labels = batch[1].to(device=CONFIG.cuda_device)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        
        optimizer.step()
        
        train_loss_sum += loss.item() * len(batch)
        train_loss_denom += len(batch)
        train_accuracy_sum += (logits.argmax(axis=1) == labels).sum().item() # how many correct predictions
        train_accuracy_denom += len(labels)

    assert train_loss_denom > 0
    assert train_accuracy_denom > 0
    train_loss = train_loss_sum / train_loss_denom
    train_accuracy = train_accuracy_sum / train_accuracy_denom
    summary_writer.add_scalar('train_loss', train_loss, epoch)
    summary_writer.add_scalar('train_accuracy', train_accuracy, epoch)
    METRICS_SUITE['train_loss'].append(train_loss)
    METRICS_SUITE['train_accuracy'].append(train_accuracy)

    lr_scheduler.step(train_loss)

    with torch.no_grad():
        test_logits = model(test_images)
        test_loss = loss_fn(test_logits, test_labels).item()
        predicted_test_labels = test_logits.argmax(axis=1)
        test_accuracy = (predicted_test_labels == test_labels).sum().item() / len(test_labels)
        summary_writer.add_scalar('test_loss', test_loss, epoch)
        summary_writer.add_scalar('test_accuracy', test_accuracy, epoch)
        METRICS_SUITE['test_loss'].append(test_loss)
        METRICS_SUITE['test_accuracy'].append(test_accuracy)
        
    summary_writer.flush()

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

## Save

In [80]:
# @launchit.disable_2
model_registry = new_model_registry()

with io.BytesIO() as b:
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'hypers': HP._asdict(),
    }, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='main', replace=True)

with io.StringIO() as b:
    json.dump(METRICS_SUITE, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

# Save Optuna trial result

In [121]:
# @launchit.disable_1
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    if 'IS_PRUNED' in optuna_trial.user_attrs:
        raise optuna.exceptions.TrialPruned()

    if not METRICS_SUITE:
        LOG(f'Empty metrics suite. Cancelling model')
        optuna_multiprocessing.save_trial_result(0)
    else:
        match LAUNCH_GOAL.goal:
            case LaunchGoal.TRAIN_MODEL_CV:
                last_std_val_accuracy = METRICS_SUITE['std_val_accuracy'][-1]
                last_mean_val_accuracy = METRICS_SUITE['mean_val_accuracy'][-1]
                    
                if last_std_val_accuracy > 0.05:
                    LOG(f'Unstable condition encountered: {last_mean_val_accuracy=}, {last_std_val_accuracy=}. Cancelling model')
                    optuna_multiprocessing.save_trial_result(0)
                else:
                    optuna_multiprocessing.save_trial_result(last_mean_val_accuracy)
                    LOG(f'Train objective result: {last_mean_val_accuracy=}')
            case LaunchGoal.TRAIN_MODEL:
                # Bad practice due to test data influences model selection
                last_test_accuracy = METRICS_SUITE['test_accuracy'][-1]
                optuna_multiprocessing.save_trial_result(last_test_accuracy)
                LOG(f'Train objective result: {last_test_accuracy=}')
            case _:
                assert False, f'Unsupported {LAUNCH_GOAL.goal=}'

# Launch creation

## TRAIN_MODEL_CV

In [25]:
# @launchit.disable
launchit_t0 = time.time()

In [11]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL_CV,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[2])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=263
Creating /home/misha/dev/mine/neurovision/denoise/s3_stacked_dae_09-launch263.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/denoise/s3_stacked_dae_09-launch263.ipynb"


## TRAIN_MODEL

In [31]:
# @launchit.disable
launchit_t0 = time.time()

In [32]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=135
Creating /home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02-launch135.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02-launch135.ipynb"


## Optuna (model selection)

### Templates

In [28]:
# @launchit.disable
# @launchit.collect_2
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            HP.fe_model_units = [
                'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2)',
                'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            initial_lr = optuna_trial.suggest_float('initial_lr', 0.0005, 0.005)
            plateau_factor = optuna_trial.suggest_float('plateau_factor', 0.1, 0.5)
            plateau_patience = optuna_trial.suggest_int('plateau_patience', HP.epochs_count // 4, HP.epochs_count // 2)
            HP.learn_rate = f'{initial_lr}, plateau(factor={plateau_factor}, patience={plateau_patience})'
            HP.cv_folds_count = 5
        case 2:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            HP.fe_model_units = [
                'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); from:s4_psd_01,450',
                'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            initial_lr = optuna_trial.suggest_float('initial_lr', 0.0001, 0.005)
            plateau_factor = optuna_trial.suggest_float('plateau_factor', 0.1, 0.5)
            plateau_patience = optuna_trial.suggest_int('plateau_patience', HP.epochs_count // 4, HP.epochs_count // 2)
            HP.learn_rate = f'{initial_lr}, plateau(factor={plateau_factor}, patience={plateau_patience})'
            HP.cv_folds_count = 5
        case 3:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            is_abs1 = optuna_trial.suggest_categorical('is_abs1', [True, False])
            is_abs2 = optuna_trial.suggest_categorical('is_abs2', [True, False])
            lcn_window1 = optuna_trial.suggest_categorical('lcn_window1', [0, 3, 5, 7])
            lcn_window2 = optuna_trial.suggest_categorical('lcn_window2', [0, 3, 5, 7])
            lcn_sigma1 = optuna_trial.suggest_int('lcn_sigma1', 1, 3)
            lcn_sigma2 = optuna_trial.suggest_int('lcn_sigma2', 1, 3)
            inject_abs = lambda flag: 'abs->' if flag else ''
            inject_lcn = lambda w, s: f'LCN({w}, {s})->' if w > 0 else ''
            HP.fe_model_units = [
                f'conv(1->32(1)x5+bias)->tanh->gain->{inject_abs(is_abs1)}{inject_lcn(lcn_window1, lcn_sigma1)}avg(2)',
                f'conv(32->64(16)x5+bias)->tanh->gain->{inject_abs(is_abs2)}{inject_lcn(lcn_window2, lcn_sigma2)}avg(2)',
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            HP.learn_rate = '0.0005, plateau(factor=0.5, patience=5)'
            HP.cv_folds_count = 5
        case _:
            assert False, f'Unsupported {study_serial=}'    

### Unleash

In [29]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL_CV:
            return ['maximize']
        case LaunchGoal.TRAIN_MODEL:
            return ['maximize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL_CV
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 3
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[2],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 25
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=2, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

[I 2026-02-26 20:13:21,180] A new study created in Journal with name: s4_cnn_02_train_model_cv_3
[I 2026-02-26 20:13:21,181] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:22:01,285] Trial 0 finished with value: 0.9907 and parameters: {'is_abs1': True, 'is_abs2': True, 'lcn_window1': 7, 'lcn_window2': 0, 'lcn_sigma1': 2, 'lcn_sigma2': 2}. Best is trial 0 with value: 0.9907.
[I 2026-02-26 20:22:01,368] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:22:59,137] Trial 1 finished with value: 0.9901833333333332 and parameters: {'is_abs1': False, 'is_abs2': False, 'lcn_window1': 7, 'lcn_window2': 3, 'lcn_sigma1': 1, 'lcn_sigma2': 2}. Best is trial 0 with value: 0.9907.
[I 2026-02-26 20:22:59,223] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:28:49,649] Trial 3 finished with value: 0.9911333333333333 and

In [30]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")

[I 2026-02-26 21:50:34,017] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.


Study statistics: 
	Number of finished trials: 25
	Number of pruned trials: 7
	Number of complete trials: 18
Best trial:
	Value: 0.9925333333333335
	Model version: 125
  Params: 
		is_abs1: True
		is_abs2: True
		lcn_window1: 3
		lcn_window2: 7
		lcn_sigma1: 2
		lcn_sigma2: 1
